# Tracking de experimentos y diagnóstico de entrenamiento

Tres flujos: (a) un run de entrenamiento con auditoría de gradientes por canal,
(b) un study de hiperparámetros con objetivo compuesto y poda, y (c) seguir o
reanudar un study desde disco. Todo queda en `.soma/` para que un front lo lea.


In [1]:
import json, pathlib
import torch
import torch.nn as nn

import soma
from soma import ChannelConfig, DifferentiableFilter, Graph, search


## (a) Run de entrenamiento con auditoría

`track_run` crea `.soma/runs/<run_id>/` (manifest con git/host, status con
heartbeat, topología del grafo, events/metrics.jsonl). `gradient_audit` con
`channels=` añade diagnósticos por canal: canales muertos, dormidos (Sokar
2023), ignorados (gradient starvation) y leakage entre grupos (CKA).


In [2]:
class Encoder(DifferentiableFilter):
    lr: float = search(1e-3, 1e-1, scale="log")

    def build_module(self, input_shape):
        return nn.Sequential(nn.Linear(input_shape[-1], 16), nn.ReLU(), nn.Linear(16, 8))

    def output_shape(self, input_shape):
        return (*input_shape[:-1], 8)

g = Graph()
g.node("encoder", Encoder())
x = torch.randn(64, 12)
y = torch.randn(64, 8)
g.materialize(x)
g.train()
g.make_optimizer(lr=0.01)


/tmp/ipykernel_1607880/2227623734.py:11: UserWarning: soma: cannot read the source of 'Encoder'; using a cloudpickle-based cache identity, which is NOT stable across Python or cloudpickle versions. Set `_cache_version = "..."` on the class for a stable cache key.
  g.node("encoder", Encoder())


Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)

In [3]:
with g.track_run("baseline", tags=["demo"]) as run:
    cfg = ChannelConfig(snapshot_every=10,
                        groups={"encoder": {"a": range(0, 4), "b": range(4, 8)}})
    with g.gradient_audit(channels=cfg) as audit:
        module = dict(g.filters())["encoder"]._module
        for epoch in range(5):
            run.log_epoch(epoch, total=5)
            with g.context() as ctx:
                g.zero_grad()
                loss = ((module(x) - y) ** 2).mean()
                g.backward(ctx, loss)   # snapshot del audit + StepCompleted
            g.step(ctx)
            run.log("loss", float(loss), step=epoch)
    print(audit.report().pretty())

run_dir = pathlib.Path(run.dir)
print(sorted(p.name for p in run_dir.iterdir()))


                  filter   steps      act|μ|       act σ      |out∂|        |θ∂|         |θ|         ∂/θ                     flags
----------------------------------------------------------------------------------------------------------------------------------
                 encoder       5   1.986e-01   2.406e-01   9.061e-02   2.791e-01   2.748e+00   1.012e-01                   HEALTHY
['diagnostics', 'events.jsonl', 'graph.json', 'graph.mmd', 'manifest.json', 'metrics.jsonl', 'status.json']


/mnt/cluster/projects/soma/soma-python/python/soma/_orchestrator.py:305: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()
/tmp/ipykernel_1607880/3521680323.py:13: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  run.log("loss", float(loss), step=epoch)


In [4]:
# Todo lo que un front necesita: eventos, métricas y diagnósticos
print((run_dir / "graph.mmd").read_text())
print(json.loads((run_dir / "status.json").read_text()))
print((run_dir / "diagnostics" / "report.json").read_text()[:400])


graph LR
    encoder[encoder]

{'state': 'completed', 'updated_at': '2026-07-29T19:08:16.485807316Z', 'heartbeat_at': '2026-07-29T19:08:16.485807316Z', 'finished_at': '2026-07-29T19:08:16.485807316Z'}
{
  "n_steps": 5,
  "filters": [
    {
      "filter": "encoder",
      "n_steps": 5,
      "metrics": {
        "act_mean_abs": 0.1986198365688324,
        "act_std": 0.2406116545200348,
        "act_zero_frac": 0.0,
        "act_zero_frac_max": 0.0,
        "act_sat_frac_max": 0.0,
        "out_grad_norm": 0.09060953557491302,
        "out_grad_max": 0.01636316627264023,
        "param_grad_norm


## (b) Study con grid, objetivo compuesto y poda

El espacio sale de los descriptores `search()` de los filtros
(`graph.search_space()`, nombres `nodo.param`). El objetivo puede ser un
callable sobre las métricas; la poda usa la regla de la mediana vía
`trial.report()`.


In [5]:
study = g.study("demo-grid", strategy="grid", n_trials=3,
                objective=lambda m: m["fit"] - 0.1 * m["cost"],
                direction="maximize", pruning=("median", 2))

def train(trial):
    g.apply_params(trial.params)
    lr = trial["encoder.lr"]
    for step in range(8):
        fit = 1.0 - abs(lr - 0.01) * 10 + step * 0.01
        if trial.report("fit", fit, step):
            return None                      # podado
    return {"fit": fit, "cost": lr * 100}

study.run(train, on_event=lambda e: print(" →", e["event_type"]))
print(study.best_trial)
print("run dir:", study.run_dir)


 → StudyStarted
 → TrialStarted
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialCompleted
 → BestUpdated
 → StudyProgress
 → TrialStarted
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric
{'id': 'trial_0000', 'params': {'encoder.lr': 0.0010000000000000002}, 'state': 'completed', 'metrics': {'fit': 0.98, 'cost': 0.10000000000000002, 'score': 0.97}, 'series': [{'name': 'fit', 'value': 0.91, 'step': 0, 'timestamp': '2026-07-29T19:08:16.515514511+00:00'}, {'name': 'fit', 'value': 0.92, 'step': 1, 'timestamp': '2026-07-29T19:08:16.515642159+00:00'}, {'name': 'fit', 'value': 0.93, 'step': 2, 'timestamp': '2026-07-29T19:08:16.515737374+00:00'}, {'name': 'fit', 'value': 0.9400000000000001, 'step': 3, 'timestamp': '2026-07-29T19:08:16.515831033+00:00'}, {'name': 'fit', 'value': 0.9500000000000001, 'step': 4, 'timestam

 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialCompleted
 → StudyProgress
 → TrialStarted
 → TrialMetric
 → TrialMetric
 → TrialMetric
 → TrialMetric


## (c) Seguir y reanudar desde disco

`study.json` se reescribe atómicamente tras cada trial: desde cualquier
máquina con acceso al directorio se puede cargar el estado, y `resume=True`
continúa exactamente donde quedó (sin repetir puntos del grid).


In [6]:
reloaded = soma.Study.load(study.run_dir)
print(reloaded.progress, len(reloaded.trials))
# reloaded.run(train, resume=True)   # continuaría si quedaran trials

for exp in soma.experiments():
    print(exp["name"], exp["metrics"], exp["tags"])


1.0 3
baseline {'loss': 0.9850006699562073} ['demo', 'run:run_20260729T190816_42e7']
demo-grid {'fit': 0.98, 'cost': 0.10000000000000002, 'score': 0.97} ['run:study_20260729T190816_2f48']
